# SIS Chatbot — QLoRA Fine-Tune (Llama 3.1 8B Instruct)

Trains a LoRA adapter on top of `meta-llama/Meta-Llama-3.1-8B-Instruct`, 4-bit NF4, using the cleaned/validated SIS dataset (`train_augmented.jsonl` / `validation.jsonl`).

**Before running:** Runtime → Change runtime type → T4 GPU (or better), then run the upload cell below and pick both `train_augmented.jsonl` and `validation.jsonl` from your machine.

On a T4 this config fits with room to spare; on anything smaller, drop `PER_DEVICE_BATCH` to 1 and raise `GRAD_ACCUM` to 32.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl huggingface_hub

In [ ]:
from google.colab import files
print("Select train_augmented.jsonl and validation.jsonl")
uploaded = files.upload()
assert "train_augmented.jsonl" in uploaded, "train_augmented.jsonl not uploaded"
assert "validation.jsonl" in uploaded, "validation.jsonl not uploaded"

In [ ]:
# mervinpraison/Llama-3.1-8B-Instruct-Tamil is apache-2.0 and NOT gated,
# so no HF token/access request is required. Login only if you plan to push
# the trained adapter to a private HF repo afterward.
# from huggingface_hub import login
# login()

In [ ]:
TRAIN_PATH = "train_augmented.jsonl"
VAL_PATH = "validation.jsonl"

BASE_MODEL = "mervinpraison/Llama-3.1-8B-Instruct-Tamil"  # apache-2.0, ungated, Tamil-tuned
OUTPUT_DIR = "sis-qlora-adapter"

# ── QLoRA config, as specified ─────────────────────────────────────────
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE = 2e-4
NUM_EPOCHS = 2
PER_DEVICE_BATCH = 2       # drop to 1 on <16GB VRAM
GRAD_ACCUM = 16            # effective batch = PER_DEVICE_BATCH * GRAD_ACCUM
MAX_SEQ_LEN = 1024
LR_SCHEDULER = "cosine"
WARMUP_RATIO = 0.03
SEED = 42

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False  # required alongside gradient checkpointing

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# `train_augmented.jsonl` / `validation.jsonl` are {"messages": [{role, content}, ...]}
# per line -- the standard SFT chat format. Deliberately NOT pre-rendered to a flat
# "text" field here: flattening with apply_chat_template() and handing SFTTrainer
# plain text makes it compute loss over every token, system prompt and the officer's
# own question included, which dilutes the gradient on the thing we actually want
# learned -- the assistant's answer. Passing the raw "messages" column lets SFTTrainer
# apply the chat template AND mask the loss to assistant turns only
# (completion_only_loss in the config below).
dataset = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": VAL_PATH})
print(dataset)
print(dataset["train"][0])

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_8bit",
    max_length=MAX_SEQ_LEN,
    packing=False,               # conversations vary in turn count -- don't pack
    completion_only_loss=True,   # mask loss to assistant turns only -- see note above.
                                 # If this errors on your installed trl version, try
                                 # assistant_only_loss=True instead (renamed across
                                 # trl releases); check with
                                 # `from trl import SFTConfig; help(SFTConfig)`.
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    bf16=True,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR} -- download this folder (or push it to a private HF repo)\n"
      f"for the evaluation and Ollama-integration steps.")

## Next steps (outside this notebook)

1. Download the `sis-qlora-adapter/` folder.
2. Run `eval_baseline_results.jsonl`'s same question set through this adapter (merge with the base model, or load with `peft`'s `PeftModel.from_pretrained`) and compare answers side-by-side against the pre-fine-tune baseline — same categories, same grading criteria.
3. Only after that comparison confirms an improvement (or at least no regression) on intent recognition / follow-ups / Tamil-Tanglish / typos / negation / refusals, integrate the adapter into the serving path. The deterministic `parse_intent` → `postgres.py` pipeline still owns every fact; this adapter only ever reaches the `general_query` LLM fallback — do not let it answer anything the deterministic layer already owns.